In [ ]:
import pandas as pd

# === Step 1: Load both CSV files ===
testing_file = "/home/dan_pham/Public/NIPTorrent/testing_samples.csv"
metadata_file = "/home/dan_pham/Public/NIPTorrent/METADATA1.csv"

testing = pd.read_csv(testing_file)
metadata = pd.read_csv(metadata_file)

# === Step 2: Standardize column names to avoid mismatch ===
testing.columns = testing.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# === Step 3: Merge based on shared identifiers ===
merged = pd.merge(
    testing,
    metadata[
        [
            "NGS RUN",
            "SAMPLE ID",
            "IONXPRESS BARCODE",
            "MATERNAL AGE",
            "GA",
            "FF (%)",
            "UNIQUE READS (M)",
            "Z21",
            "Z18",
            "Z13",
            "GENDER",
            "AVERAGE READ LENGTH (bp)",
            "GC CONTENT (%)",
            "DUPLICATION (%)",
            "Nuchal Translucency (mm)",
        ]
    ],
    on=["NGS RUN", "SAMPLE ID", "IONXPRESS BARCODE"],
    how="left",
    suffixes=("", "_meta"),
)

# === Step 4: Fill missing columns in testing_samples with metadata values ===
for col in [
    "MATERNAL AGE",
    "GA",
    "FF (%)",
    "UNIQUE READS (M)",
    "Z21",
    "Z18",
    "Z13",
    "GENDER",
    "AVERAGE READ LENGTH (bp)",
    "GC CONTENT (%)",
    "DUPLICATION (%)",
    "Nuchal Translucency (mm)",
]:
    merged[col] = merged[col].combine_first(merged[f"{col}_meta"])

# === Step 5: Drop extra columns ===
merged = merged[
    [c for c in merged.columns if not c.endswith("_meta")]
]

# === Step 6: Save output ===
output_path = r"C:\Users\ariha.danph\Downloads\NIPTorrent\testing_samples_filled.csv"
merged.to_csv(output_path, index=False)

print(f"✅ Filled data saved to: {output_path}")


✅ Filled data saved to: C:\Users\ariha.danph\Downloads\NIPTorrent\testing_samples_filled.csv


In [ ]:
import pandas as pd

# === Step 1: Load file ===
file_path = "/home/dan_pham/Public/NIPTorrent/test.csv"
df = pd.read_csv(file_path)

# === Step 2: Extract unique values ===
col1 = set(df['sample_test1'].dropna().astype(str).str.strip())
col2 = set(df['sample_test2'].dropna().astype(str).str.strip())

# === Step 3: Compare and find differences ===
to_delete = sorted(list(col1 - col2))  # in col1 but not in col2
to_add = sorted(list(col2 - col1))      # in col2 but not in col1

# === Step 4: Save to CSV files ===
pd.DataFrame({'to_delete': to_delete}).to_csv("/home/dan_pham/Public/NIPTorrent/delete.csv", index=False)
pd.DataFrame({'to_add': to_add}).to_csv("/home/dan_pham/Public/NIPTorrent/add.csv", index=False)

print("✅ Done!")
print(f"Samples in col1 but not in col2: {len(to_delete)} → delete.csv")
print(f"Samples in col2 but not in col1: {len(to_add)} → add.csv")


✅ Done!
Samples in col1 but not in col2: 1 → delete.csv
Samples in col2 but not in col1: 90 → add.csv


In [ ]:
library(readr)
library(dplyr)

# File paths
gold_file <- "/home/dan_pham/Public/NIPTorrent/testing_samples.csv"
result_file <- "/home/dan_pham/Downloads/RESULTS/sample_run/nipt_risk_without_blacklist/nipt_risk/results.csv"

# Output path
output_file <- "/home/dan_pham/Public/NIPTorrent/testing_samples_matched.csv"

# Read files
gold <- read_csv(gold_file, show_col_types = FALSE)
result <- read_csv(result_file, show_col_types = FALSE)

# Column names
gold_id <- names(gold)[1]
result_id <- names(result)[1]

# Match by sample names
matched <- gold %>%
    filter(.data[[gold_id]] %in% result[[result_id]])

# Save output
write_csv(matched, output_file)

cat("Done! Matched rows saved to:", output_file, "\n")


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)

# Load file
file_path <- "/home/dan_pham/Downloads/result.csv"
df <- read_csv(file_path, show_col_types = FALSE)

# Function to convert Z-score to Positive/Negative
classify <- function(z) {
    if (is.na(z)) return(NA)
    if (z > 3 | z < -3) return("Positive")
    return("Negative")
}

# Create predicted vs gold labels
df$Z21_test <- sapply(df[[2]], classify)  # col B
df$Z18_test <- sapply(df[[3]], classify)  # col C
df$Z13_test <- sapply(df[[4]], classify)  # col D

df$Z21_gold <- sapply(df[[8]], classify)  # col H
df$Z18_gold <- sapply(df[[9]], classify)  # col I
df$Z13_gold <- sapply(df[[10]], classify) # col J

# Function to compute confusion matrix manually
make_cm <- function(pred, ref) {
    table(
        Prediction = factor(pred, levels=c("Positive","Negative")),
        Reference  = factor(ref, levels=c("Positive","Negative"))
    )
}

cm21 <- make_cm(df$Z21_test, df$Z21_gold)
cm18 <- make_cm(df$Z18_test, df$Z18_gold)
cm13 <- make_cm(df$Z13_test, df$Z13_gold)

print("Z21 Confusion Matrix:")
print(cm21)
print("Z18 Confusion Matrix:")
print(cm18)
print("Z13 Confusion Matrix:")
print(cm13)

# Plot confusion matrix
plot_cm <- function(cm, title) {
    m <- as.data.frame(cm)
    ggplot(m, aes(Prediction, Reference, fill = Freq)) +
        geom_tile(color="black") +
        geom_text(aes(label = Freq), size = 7) +
        scale_fill_gradient(low="white", high="steelblue") +
        ggtitle(title) +
        theme_minimal(base_size = 16)
}

# Draw plots
plot_cm(cm21, "Confusion Matrix – Z21")
plot_cm(cm18, "Confusion Matrix – Z18")
plot_cm(cm13, "Confusion Matrix – Z13")


Loading required package: ggplot2

Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Loading required package: lattice



ERROR: Error: package or namespace load failed for ‘caret’ in loadNamespace(j <- i[[1L]], c(lib.loc, .libPaths()), versionCheck = vI[[j]]):
 namespace ‘parallelly’ 1.36.0 is being loaded, but >= 1.44.0 is required
